# Luminarios del arrecife astral
GAN de personajes originales. Se utiliza una DCGAN de 64 ? 64. El conjunto se obtiene mediante dibujos param?tricos propios. La galer?a se genera exclusivamente con los pesos entrenados.


## Universo y datos
# Luminarios del arrecife astral

## Universo visual

El arrecife astral es un mundo oscuro habitado por criaturas pequeñas que producen luz. Los personajes tienen cuerpos redondos, ojos claros, antenas y patas cortas. Se usan tonos turquesa, azul, violeta, naranja, verde y rosa sobre un fondo azul oscuro. La galería se evalúa por la presencia de un cuerpo reconocible, rasgos visibles y diferencias de color y forma entre personajes.

## Origen y permiso de uso

Se construyó un conjunto original mediante `build_dataset.py`, con dibujos de figuras geométricas realizadas con Pillow. No se descargaron personajes, fotografías ni imágenes de terceros. El conjunto se creó dentro del proyecto para su uso en entrenamiento y evaluación. No se atribuye una licencia externa a estas imágenes ni se afirma una cesión de derechos de terceros.

El dibujo paramétrico se utiliza únicamente para preparar los datos de entrenamiento. La galería final se produce con el generador de la GAN y sus pesos guardados; no se obtiene del programa de dibujo.

## Composición

- 1,536 imágenes RGB distintas, de 64 × 64 píxeles.
- Semilla del conjunto: 20261011.
- Tres formas de cuerpo: ovalado, redondeado y campana.
- Uno, dos o tres ojos; dos, tres o cuatro patas.
- Antena simple, doble, cuernos o ausencia de antenas.
- Seis colores principales con pequeñas variaciones.
- Variaciones de posición, tamaño, boca y puntos luminosos.

Cada dibujo se realiza a 128 × 128 y se reduce a 64 × 64 mediante Lanczos. El manifiesto registra los rasgos y la huella de los píxeles de cada imagen. No se utilizan rostros de personas reales ni personajes comerciales.

## Preparación

`prepare_data.py` convierte las imágenes a RGB, aplica recorte cuadrado centrado, ajusta la resolución y elimina duplicados exactos después de la transformación. El archivo `data_report.json` registra entradas, imágenes conservadas y descartes.

Durante el entrenamiento se transforma cada canal desde [0, 1] a [-1, 1]. No se aplica aumento de datos en las tres corridas controladas. El fondo y el encuadre son constantes para reducir la dificultad del aprendizaje en CPU.

## Limitaciones

El conjunto representa una familia de dibujos simples, no la variedad de un universo artístico completo. Las combinaciones de rasgos se originan en reglas conocidas. Aprender esta distribución no demuestra que el modelo comprenda historias, roles o anatomía. El fondo uniforme puede dominar las distancias en píxeles; por ello también se informa una distancia sobre la unión de las regiones de primer plano.

La evaluación busca novedad respecto a este conjunto. Una distancia alta no basta para concluir que una imagen es un personaje útil: también puede indicar ruido o defectos.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content/drive/MyDrive/gan-personajes
!pip install -r requirements.txt


## Hip?tesis y protocolo previos
# Experimentos controlados

## Protocolo registrado antes del entrenamiento

Se fijan 1,536 dibujos del mismo conjunto, 60 épocas, lote de 64, semilla 42, ruido de evaluación con semilla 1234 y 16 vectores fijos. El generador y el discriminador usan 16 canales iniciales. El vector latente tiene 64 componentes. Se utiliza Adam con tasa 0.0002 para ambas redes, beta1 0.5 y beta2 0.999. No se aplica aumento de datos.

Cada experimento se compara contra la base. El cambio de pérdida y la normalización espectral no se combinan. El orden de los lotes y el ruido de entrenamiento tienen generadores aleatorios independientes, por lo que la variante espectral no cambia esas secuencias. También se conserva la misma inicialización de los pesos originales antes de aplicar normalización espectral.

## Base: BCE no saturante

Se utiliza BCE sobre logits. El discriminador distingue imágenes reales y generadas. El generador intenta que las imágenes falsas reciban la etiqueta real. Se conserva esta configuración como referencia, sin afirmar que una pérdida baja demuestre buena calidad.

## Experimento de pérdida: hinge

**Hipótesis previa:** Se espera que la pérdida hinge mejorará la diversidad de los personajes porque utiliza un margen para el discriminador y una señal lineal para el generador. Lo consideraremos útil si la diversidad RMS media de las últimas tres rejillas no cae más de 10 % frente a la base y la inspección del mismo ruido fijo muestra cuerpos y rasgos al menos igual de reconocibles, sin repetición dominante de una sola forma.

Único cambio: `--loss hinge`. El discriminador minimiza `mean(relu(1-D(real))) + mean(relu(1+D(fake)))`. El generador minimiza `-mean(D(fake))`. Las pérdidas hinge y BCE tienen escalas distintas; no se comparan sus valores absolutos como una medida de calidad.

## Experimento de estabilización: normalización espectral

**Hipótesis previa:** Se espera que la normalización espectral mejorará la estabilidad porque limita la amplificación de las convoluciones del discriminador. Lo consideraremos útil si la desviación de la pérdida de D durante las últimas 20 épocas es menor que en la base, la diversidad media de las últimas tres rejillas no cae más de 10 % y la inspección visual no muestra una pérdida clara de rasgos reconocibles.

Único cambio: `--spectral`. Se conservan BCE, BatchNorm y el resto de la configuración. La técnica controla los pesos de las convoluciones, pero no impone una cota global a toda la red con BatchNorm.

## Medición e interpretación

La diversidad RMS se calcula entre las 16 imágenes de ruido fijo. Es una señal de diferencias en píxeles, no una medida directa de variedad de personajes. Puede subir por ruido o defectos. Por eso se evalúa junto con las rejillas y la galería.

La desviación de la pérdida se usa como un indicador limitado de oscilación. Una curva suave también puede corresponder a un modelo que aprende poco. Las decisiones finales requieren revisar las figuras.

Los resultados se registran en `resultados.md` una vez finalizadas las corridas. La hipótesis queda separada de las observaciones posteriores.

## Fuentes

- Goodfellow et al. (2014), [Generative Adversarial Nets](https://arxiv.org/abs/1406.2661).
- Radford et al. (2015), [Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks](https://arxiv.org/abs/1511.06434).
- Miyato et al. (2018), [Spectral Normalization for Generative Adversarial Networks](https://arxiv.org/abs/1802.05957).


In [ ]:
from pathlib import Path
import subprocess, sys
if not any(Path('data/raw/luminarios').glob('*.png')):
    subprocess.run([sys.executable, 'build_dataset.py'], check=True)
if not any(Path('data/processed/all').glob('*.png')):
    subprocess.run([sys.executable, 'prepare_data.py', '--raw', 'data/raw/luminarios', '--out', 'data/processed', '--size', '64'], check=True)


In [ ]:
subprocess.run([sys.executable, 'run_experiments.py', '--epochs', '60'], check=True)


In [ ]:
from IPython.display import Image, display
for name in ['base', 'exp_loss_hinge', 'exp_stab_sn']:
    subprocess.run([sys.executable, 'analysis.py', 'curves', f'runs/{name}'], check=True)
    subprocess.run([sys.executable, 'analysis.py', 'evolution', f'runs/{name}', '--epochs', '0', '5', '20', '40', '60'], check=True)
    display(Image(f'runs/{name}/curvas.png'))
    display(Image(f'runs/{name}/evolucion.png'))
display(Image('comparacion.png'))


## Registro adicional de estabilizaci?n

La hip?tesis se registr? en `estadisticas.md` antes de ejecutar esta variante. Se incorpora al notebook como parte del informe.

# Regularización adicional por estadísticas de los lotes

## Observación previa

Las tres corridas controladas de 60 épocas terminaron. La base y hinge mantienen ruido visible. La variante espectral mejora la variedad, pero conserva defectos. Durante las primeras 30 épocas de la variante bilineal se observaron cuerpos suaves con poco contraste y una pérdida de variedad.

## Hipótesis previa

Se espera que comparar la media y la variación de los lotes mejorará el fondo, el color y la diversidad porque obliga al generador a aproximar estadísticas del conjunto real, además de responder al discriminador. Lo consideraremos útil si el fondo oscuro se recupera, aparecen cuerpos y rasgos reconocibles y se pueden seleccionar diez muestras con diferencias visibles entre 200 candidatos.

## Método

Se reduce cada lote real y generado a 8 × 8 mediante promedio. Se calcula la diferencia absoluta media entre las medias de ambos lotes y entre sus desviaciones por canal y posición. Se agrega diez veces esa suma a la pérdida adversarial hinge de G.

`L_G = -mean(D(G(z))) + 10 × (error de medias + error de desviaciones)`

No se empareja cada muestra generada con una imagen real. Se comparan estadísticas del lote completo. La galería sigue siendo una salida del generador entrenado; no se dibujan ni corrigen personajes en la etapa de generación.

Se conservan los datos, las semillas y la configuración de la variante bilineal: 120 épocas, lote 64, canales 16, tasa G 0.0002, tasa D 0.0001, normalización espectral y ausencia de BatchNorm en D. Se utiliza un nuevo entrenamiento desde la inicialización, sin cargar pesos de las corridas anteriores.

## Alcance

Es una regularización propia sobre estadísticas de píxeles a baja resolución. La idea general de comparar estadísticas se relaciona con las técnicas de estabilización de Salimans et al. (2016), pero no reproduce su método de características aprendidas ni garantiza calidad semántica. Las medias y desviaciones pueden producir imágenes suaves y no aseguran ojos, patas o anatomía correcta.

La corrida se evalúa aparte de los tres experimentos obligatorios. La pérdida total de G incluye dos términos y no se compara directamente con el valor hinge de las otras corridas. `regularization.csv` conserva ambos términos por separado.

## Ejecución

```bash
python train.py --data data/processed --name moment_refined --epochs 120 --g_arch resize --loss hinge --spectral --d_norm none --lr_d 0.0001 --moment_weight 10 --threads 2
```

## Referencia

Salimans et al. (2016), [Improved Techniques for Training GANs](https://arxiv.org/abs/1606.03498).


In [ ]:
if not Path('runs/moment_refined/ckpt/G_final.pt').exists():
    args = [sys.executable, 'train.py', '--data', 'data/processed', '--name', 'moment_refined', '--epochs', '120', '--g_arch', 'resize', '--loss', 'hinge', '--spectral', '--d_norm', 'none', '--lr_d', '0.0001', '--moment_weight', '10', '--threads', '2']
    if Path('runs/moment_refined/ckpt/last.pt').exists():
        args.append('--resume')
    subprocess.run(args, check=True)
subprocess.run([sys.executable, 'collect_results.py'], check=True)


## Resultados y selecci?n
La interpretaci?n de las corridas se encuentra en `resultados.md`. La selecci?n final, el checkpoint, las semillas y los criterios se conservan en `galeria/seleccion.json`.


# Variante adicional de calidad

## Motivo

La base DCGAN de 60 épocas produjo textura con mucho ruido y poca variedad. El discriminador mantuvo una pérdida muy baja durante una gran parte del entrenamiento. La variante hinge comenzó a mostrar cuerpos, pero la diversidad cayó entre las épocas 10 y 30. Estas observaciones justifican probar una variante completa para la galería, además de los experimentos controlados.

## Hipótesis registrada antes de ejecutar la variante

Se espera que un generador con ampliación bilineal seguida de convoluciones, junto con un discriminador espectral sin BatchNorm y una tasa menor para D, mejorará la calidad porque reduce los patrones de tablero y limita la ventaja del discriminador. Lo consideraremos útil si, entre 200 candidatos, se pueden seleccionar al menos 10 cuerpos reconocibles con diferencias visibles de forma o color, y las rejillas de ruido fijo muestran variedad sostenida.

## Configuración

Se conservan las 1,536 imágenes, el vector latente de 64 componentes, los canales iniciales de 16 y las semillas. Se realizan 120 épocas con hinge, normalización espectral, ausencia de BatchNorm en D y tasas 0.0002 para G y 0.0001 para D. El generador conserva BatchNorm y Tanh, pero sustituye las cuatro ampliaciones de convolución transpuesta por ampliación bilineal y convolución de 3 × 3.

Esta corrida cambia varias condiciones. Se considera una variante completa de calidad, no un experimento que permita atribuir el efecto a una sola técnica. Los requisitos de comparación de pérdida y estabilización se evalúan únicamente con las tres corridas controladas de 60 épocas.

## Comando

```bash
python train.py --data data/processed --name refined --epochs 120 --g_arch resize --loss hinge --spectral --d_norm none --lr_d 0.0001
```

El resultado posterior y la decisión sobre su uso en la galería se registran en `resultados.md`.


In [ ]:
if not Path('runs/refined/ckpt/G_final.pt').exists():
    args = [sys.executable, 'train.py', '--data', 'data/processed', '--name', 'refined', '--epochs', '120', '--g_arch', 'resize', '--loss', 'hinge', '--spectral', '--d_norm', 'none', '--lr_d', '0.0001', '--threads', '2']
    if Path('runs/refined/ckpt/last.pt').exists():
        args.append('--resume')
    subprocess.run(args, check=True)


In [ ]:
import json
selection_path = Path('galeria/seleccion.json')
if selection_path.exists():
    subprocess.run([sys.executable, 'generate_gallery.py', '--from_selection', str(selection_path)], check=True)
    display(Image('galeria/galeria.png'))
else:
    subprocess.run([sys.executable, 'generate_gallery.py', '--ckpt', 'runs/base/ckpt/G_final.pt', '--n', '200'], check=True)
    display(Image('evidencias/candidatos.png'))


In [ ]:
if selection_path.exists():
    subprocess.run([sys.executable, 'nn_test.py', '--data', 'data/processed'], check=True)
    display(Image('nn_resultados/vecinos.png'))
    display(Image('nn_resultados/calibracion.png'))
    print(Path('nn_resultados/resumen.json').read_text())


## Informe
`presentacion.md` contiene el informe organizado en 12 diapositivas, la matriz de evidencias y la reflexi?n final. `README.md` contiene los comandos de ejecuci?n y las versiones utilizadas.
